# Denoising Autoencoder — Extracting and Composing Robust Features with Denoising Autoencoders (2008)

_Papers · Self-supervised & Representation_

**Corrupt the input on purpose, then train the autoencoder to rebuild the clean original — forcing it to learn robust features instead of copying the identity.**

---

This notebook is a practice scaffold for the **Denoising Autoencoder — Extracting and Composing Robust Features with Denoising Autoencoders (2008)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: denoising loss scored against the CLEAN input. ---
x_clean = torch.tensor([0.9, 0.2, 0.8, 0.5])           # clean target
mask    = torch.tensor([1., 1., 0., 1.])               # masking noise: pixel 3 zeroed (nu = 1/4)
x_tilde = x_clean * mask                                # corrupted encoder input = [0.9,0.2,0.0,0.5]
z       = torch.tensor([0.85, 0.25, 0.6, 0.5])         # a decoder reconstruction
sse = ((x_clean - z)**2).sum()                         # ERROR vs CLEAN x, not vs x_tilde
mse = ((x_clean - z)**2).mean()
print("worked example:  x_tilde =", x_tilde.tolist())
print("  sum of squared errors =", round(sse.item(), 5),
      " MSE =", round(mse.item(), 5),
      " masked-pixel cost =", round(((x_clean[2]-z[2])**2).item(), 4))
# worked example:  x_tilde = [0.9, 0.2, 0.0, 0.5]
#   sum of squared errors = 0.045   MSE = 0.01125   masked-pixel cost = 0.04


# --- 1. The (denoising) autoencoder. nu=0 -> plain AE (paper's SAA-3); nu>0 -> denoising AE (SdA). ---
class DAE(nn.Module):
    def __init__(self, H=512):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, H), nn.ReLU())   # over-complete code allowed (H may be >784)
        self.dec = nn.Sequential(nn.Linear(H, 784), nn.Sigmoid())
    def encode(self, x): return self.enc(x)                      # the features we keep + probe
    def forward(self, x): return self.dec(self.enc(x))

def corrupt(x, nu):                                              # masking noise: zero a fraction nu of pixels
    if nu == 0: return x
    keep = (torch.rand_like(x) > nu).float()                    # fresh random mask EACH call
    return x * keep


# --- 2. An MNIST subset (torchvision, preinstalled). Flatten to 784, values in [0,1]. ---
tf  = T.Compose([T.ToTensor(), T.Lambda(lambda t: t.view(-1))])
tr  = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
te  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
ridx = np.random.RandomState(0).permutation(len(tr))[:8000]
tidx = np.random.RandomState(1).permutation(len(te))[:2000]
Xtr = torch.stack([tr[i][0] for i in ridx]).to(device); ytr = torch.tensor([tr[i][1] for i in ridx])
Xte = torch.stack([te[i][0] for i in tidx]).to(device); yte = torch.tensor([te[i][1] for i in tidx])


def train_dae(nu, epochs=30, lr=1e-3, B=256):
    torch.manual_seed(0)
    ae = DAE().to(device); opt = torch.optim.Adam(ae.parameters(), lr=lr); loss_fn = nn.MSELoss()
    ae.train()
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr)); tot = 0.0; nb = 0
        for s in range(0, len(Xtr), B):
            xb = Xtr[perm[s:s+B]]
            xc = corrupt(xb, nu)                                # encoder sees the CORRUPTED input
            z  = ae(xc); loss = loss_fn(z, xb)                  # but the target is the CLEAN xb
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if ep % 10 == 0: print(f"  [nu={nu}] epoch {ep:2d}  recon MSE {tot/nb:.4f}")
    return ae

print("\n=== plain autoencoder (nu=0, = paper's SAA-3) ===");    ae_plain = train_dae(nu=0.0)
print("\n=== denoising autoencoder (nu=0.3, = paper's SdA) ==="); ae_dae   = train_dae(nu=0.3)


# --- 3. Denoise demo: feed CORRUPTED held-out digits through the denoising AE; z recovers the clean digit. ---
with torch.no_grad():
    x_demo  = Xte[:8]; x_corr = corrupt(x_demo, 0.3)
    x_recon = ae_dae(x_corr)
    recon_err = nn.functional.mse_loss(x_recon, x_demo).item()
print(f"\ndenoise demo: MSE(reconstruction, CLEAN held-out) = {recon_err:.4f}  (it fills in the zeroed pixels)")


# --- 4. Ablation: feature quality via a LINEAR PROBE on the FROZEN encoder (clean inputs at eval). ---
def linear_probe(ae, epochs=60, lr=1e-2):
    torch.manual_seed(0)
    with torch.no_grad():
        Ztr = ae.encode(Xtr); Zte = ae.encode(Xte)             # frozen codes of CLEAN inputs
    clf = nn.Linear(Ztr.shape[1], 10).to(device)               # a single linear layer = linear probe
    opt = torch.optim.Adam(clf.parameters(), lr=lr); ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad(); ce(clf(Ztr), ytr.to(device)).backward(); opt.step()
    with torch.no_grad():
        acc = (clf(Zte).argmax(1).cpu() == yte).float().mean().item()
    return acc

acc_plain = linear_probe(ae_plain)
acc_dae   = linear_probe(ae_dae)
print("\nlinear-probe accuracy on frozen codes (feature quality):")
print(f"  plain autoencoder     (nu=0.0): {acc_plain:.3f}")
print(f"  denoising autoencoder (nu=0.3): {acc_dae:.3f}")
print("Denoising features separate digits better -> SdA beats SAA (paper Table 1), at toy scale.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does corrupting the input (masking noise) and reconstructing the CLEAN original make an autoencoder's features more useful than a plain autoencoder's — measured by a linear probe?_

In [ ]:
import torch, torch.nn as nn, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)

# Plain (nu=0) vs denoising (nu>0) autoencoder feature quality on an MNIST subset (toy reproduction of
# the paper's SdA-3 > SAA-3 result), measured by a linear probe on the frozen encoder code.
class DAE(nn.Module):
    def __init__(s, H=512):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(784, H), nn.ReLU())
        s.dec = nn.Sequential(nn.Linear(H, 784), nn.Sigmoid())
    def encode(s, x): return s.enc(x)
    def forward(s, x): return s.dec(s.enc(x))

def corrupt(x, nu):
    return x if nu == 0 else x * (torch.rand_like(x) > nu).float()   # fresh masking noise each call

tf  = T.Compose([T.ToTensor(), T.Lambda(lambda t: t.view(-1))])
tr  = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
te  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
ri  = np.random.RandomState(0).permutation(len(tr))[:8000]
ti  = np.random.RandomState(1).permutation(len(te))[:2000]
Xtr = torch.stack([tr[i][0] for i in ri]); ytr = torch.tensor([tr[i][1] for i in ri])
Xte = torch.stack([te[i][0] for i in ti]); yte = torch.tensor([te[i][1] for i in ti])

def train(nu, epochs=30, B=256):
    torch.manual_seed(0); ae = DAE(); opt = torch.optim.Adam(ae.parameters(), 1e-3); L = nn.MSELoss()
    last = 0.0
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr)); tot=0.; nb=0
        for s in range(0, len(Xtr), B):
            xb = Xtr[perm[s:s+B]]; loss = L(ae(corrupt(xb, nu)), xb)   # target = clean xb
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
        last = tot/nb
    return ae, last

def probe(ae, epochs=60):
    torch.manual_seed(0)
    with torch.no_grad(): Ztr, Zte = ae.encode(Xtr), ae.encode(Xte)
    clf = nn.Linear(Ztr.shape[1], 10); opt = torch.optim.Adam(clf.parameters(), 1e-2); ce = nn.CrossEntropyLoss()
    for _ in range(epochs): opt.zero_grad(); ce(clf(Ztr), ytr).backward(); opt.step()
    with torch.no_grad(): return (clf(Zte).argmax(1) == yte).float().mean().item()

ae_plain, mse_plain = train(nu=0.0)
ae_dae,   mse_dae   = train(nu=0.3)
print("plain AE  (nu=0.0):  probe acc", round(probe(ae_plain),3), " recon MSE", round(mse_plain,4))
print("denoise AE(nu=0.3):  probe acc", round(probe(ae_dae),  3), " recon MSE", round(mse_dae,4))
# Denoising features win the probe even though the plain AE has lower reconstruction MSE (it nearly copies).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline (better features). You trained two single-layer autoencoders on MNIST that differ
            only in $\nu$ (one $\nu=0$, one $\nu=0.3$), froze each encoder, and trained a linear probe on its
            code. The denoising encoder's probe scores higher. What does this demonstrate, and which mechanism is
            responsible?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note both probes are the SAME one-layer linear classifier on the SAME-size frozen code; only the encoder's training differed. — _Holding the probe fixed makes the accuracy gap a pure measure of feature (representation) quality._
- Recall the $\nu=0$ encoder could copy the input through (identity), so it never had to learn pixel dependencies. — _With nothing forcing structure, its code is close to a re-encoding of raw pixels — poorly separable._
- Recall the $\nu=0.3$ encoder had to fill in zeroed pixels from surviving ones, forcing it to learn strokes/shapes. — _Those dependency-capturing features are linearly more useful for telling digits apart._

**Answer:** It demonstrates the paper's central claim: corrupting the input and training to reconstruct the
                 clean original makes the encoder learn more useful (robust) features than a plain
                 autoencoder — features good enough that even a linear probe separates digits better. The
                 responsible mechanism is the denoising criterion (Eq. 5): because the network must predict
                 missing pixels from surviving ones, it is forced to capture the data's structure (the
                 dependencies among pixels) instead of copying the identity. This is the table-level result
                 (SdA-3 beats SAA-3) reproduced at toy scale. Our CODEVIZ shows the denoising probe accuracy above
                 the plain-AE probe accuracy.

</details>

**Problem 2.** Your worked example gave denoising loss $\text{MSE}=0.01125$ for clean $x=[0.9,0.2,0.8,0.5]$,
            mask zeroing pixel 3, and reconstruction $z=[0.85,0.25,0.6,0.5]$. Suppose after more training the
            decoder improves its guess of the missing pixel to $z=[0.85,0.25,0.78,0.5]$. What is the new MSE, and
            what did training mostly improve?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Per-pixel errors vs CLEAN $x$: $x-z=[0.05,\ -0.05,\ 0.02,\ 0]$. — _Only pixel 3's guess changed ($0.6 \to 0.78$, vs the true $0.8$); it is now nearly right._
- Square and sum: $[0.0025,\ 0.0025,\ 0.0004,\ 0] \to 0.0054$. — _The masked pixel's squared error fell from $0.04$ to $0.0004$._
- Divide by 4 pixels: $\text{MSE}=0.0054/4=0.00135$. — _Mean over the 4 pixels._

**Answer:** The new $\text{MSE}=0.00135$, down from $0.01125$ — about $8\times$ smaller. Almost the entire
                 improvement came from the masked pixel (its squared error dropped from $0.04$ to $0.0004$):
                 the network got better at guessing the missing value from its neighbours, which is exactly
                 the skill the denoising objective rewards. The visible pixels were already nearly correct and
                 barely moved.

</details>

**Problem 3.** Ablation (the paper's own). You set $\nu=0$ — no corruption — and retrain, everything else
            identical. This is the plain autoencoder (the paper's "SAA-3 = SdA-3 with $\nu=0\%$"). With an
            over-complete code, its reconstruction loss goes very low but its linear-probe accuracy is poor. Why
            both at once?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With $\nu=0$ and a wide code, the network can learn $g(f(x)) \approx x$ — essentially the identity. — _Nothing forbids copying when the input is clean and the code is not a bottleneck._
- Identity gives near-zero reconstruction loss but a code that is just a re-encoding of raw pixels. — _Low loss does not imply useful features — it can be achieved by copying._
- A raw-pixel-like code is not linearly separable by class, so the probe does poorly. — _No structure (strokes, shapes) was forced into the representation._

**Answer:** Because low reconstruction loss does not mean good features. At $\nu=0$ with an
                 over-complete code the autoencoder can reach near-zero loss by learning the identity
                 (copying the input), so the loss looks great — but the code carries no learned structure, just a
                 re-encoded copy of the pixels, which a linear probe cannot separate by digit. Turning corruption
                 on ($\nu\gt0$) kills the identity shortcut and forces the encoder to learn the pixel dependencies
                 that are linearly useful. This is precisely why the paper measures success by downstream
                 classification, not reconstruction loss — and why SdA-3 beats SAA-3 in Table 1. Our CODEVIZ
                 ablation bar shows the $\nu=0$ probe accuracy dropping below the denoising one despite comparable
                 (or lower) reconstruction loss.

</details>